<a href="https://colab.research.google.com/github/Yatha04/ML-tinkering/blob/main/HopefieldNetwork.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import numpy as np
import warnings

In [3]:
warnings.filterwarnings("ignore")

In [5]:
#Step 1: Convert binary pattern (0's and 1's to a bipolar pattern +1/-1's)
def to_bipolar(pattern):
  return np.where(np.array(pattern) == 0,-1, 1)

In [6]:
def to_binary(pattern):
  return np.where(np.array(pattern) == 1,1,0)

In [19]:
#Step 2: Build weight matrix using Hebbian Learning(AKA outer product rule)
def train(pattern):
  N = len(pattern[0])
  W = np.zeros((N,N))

  for p in pattern:
    p = np.array(p).reshape(-1,1)
    W += np.dot(p,p.T)

    W = W/N
    np.fill_diagonal(W,0)
  return W

In [10]:
def energy(state, W):
  s = np.array(state)
  return -0.5 * s @ W @ s


In [11]:
def update_neuron(state, W, i):
    net_input = np.dot(W[i], state)     # weighted sum of all inputs to neuron i

    if net_input > 0:
        return 1
    elif net_input < 0:
        return -1
    else:
        return state[i]

In [21]:
#run the recall loop:

def recall(W, noisy_input, max_iter = 20):
    state = np.array(noisy_input, dtype=float)
    N = len(state)
    history = [(state.copy(), energy(state, W))]

    for iteration in range(max_iter):
        changed = False
        update_order = np.random.permutation(N)     # random neuron order

        for i in update_order:
            new_val = update_neuron(state, W, i)
            if new_val != state[i]:
                state[i] = new_val
                changed = True

        history.append((state.copy(), energy(state, W)))

        if not changed:
            return state, history, True             # converged!

    return state, history, False

In [12]:
def similarity(a, b):
    a, b = np.array(a), np.array(b)
    return np.dot(a, b) / len(a)

In [14]:
def add_noise(pattern, noise_level=0.2, seed=None):
    """
    Randomly flip a fraction of bits in a pattern to simulate
    a corrupted or partial input.

    noise_level = 0.0 → no corruption
    noise_level = 0.5 → half the bits flipped (50% noise)
    noise_level = 1.0 → all bits flipped (opposite pattern)
    """
    if seed is not None:
        np.random.seed(seed)

    pattern = np.array(pattern, dtype=float)
    flip_mask = np.random.random(len(pattern)) < noise_level
    noisy = pattern.copy()
    noisy[flip_mask] *= -1              # flip selected bits

    return noisy

In [16]:
def display_pattern(pattern, width, title=""):
    """
    Render a 1D pattern as a 2D grid for visualization.
    Uses █ for +1 (active) and · for -1 (inactive).
    """
    if title:
        print(f"\n  {title}")
        print("  " + "─" * (width * 2))

    pattern = np.array(pattern)
    for row_start in range(0, len(pattern), width):
        row = pattern[row_start : row_start + width]
        print("  " + "  ".join("█" if v > 0 else "·" for v in row))

    print()

In [22]:
if __name__ == "__main__":

    print("=" * 55)
    print("  HOPFIELD NETWORK DEMO")
    print("=" * 55)

    # Define three 5×5 patterns (25 neurons each).
    # We use simple geometric shapes: a cross, a diagonal, a block.

    W = 5   # grid width
    H = 5   # grid height
    N = W * H  # total neurons = 25

    pattern_cross = to_bipolar([
        0, 0, 1, 0, 0,
        0, 0, 1, 0, 0,
        1, 1, 1, 1, 1,
        0, 0, 1, 0, 0,
        0, 0, 1, 0, 0,
    ])

    pattern_x = to_bipolar([
        1, 0, 0, 0, 1,
        0, 1, 0, 1, 0,
        0, 0, 1, 0, 0,
        0, 1, 0, 1, 0,
        1, 0, 0, 0, 1,
    ])

    pattern_box = to_bipolar([
        1, 1, 1, 1, 1,
        1, 0, 0, 0, 1,
        1, 0, 0, 0, 1,
        1, 0, 0, 0, 1,
        1, 1, 1, 1, 1,
    ])

    stored_patterns = [pattern_cross, pattern_x, pattern_box]
    pattern_names   = ["Cross", "X shape", "Box"]

    # ── Print stored patterns ──────────────────────────────────
    print("\n📦 STORING PATTERNS:")
    for name, p in zip(pattern_names, stored_patterns):
        display_pattern(p, W, title=name)

    # ── Train the network ──────────────────────────────────────
    weights = train(stored_patterns)

    print(f"Weight matrix shape: {weights.shape}")
    print(f"Network capacity: ~{int(0.138 * N)} patterns for {N} neurons")
    print(f"Patterns stored: {len(stored_patterns)}")

    capacity_ok = len(stored_patterns) <= 0.138 * N
    print(f"Within capacity limit: {'✓ Yes' if capacity_ok else '✗ No (may get recall errors)'}")

    # ── Test recall with noise ─────────────────────────────────
    print("\n" + "=" * 55)
    print("  RECALL TEST")
    print("=" * 55)

    for noise_level in [0.1, 0.2, 0.4]:
        print(f"\n── Noise level: {noise_level * 100:.0f}% ──────────────────────")

        for name, original in zip(pattern_names, stored_patterns):
            noisy = add_noise(original, noise_level=noise_level, seed=42)
            recalled, history, converged = recall(weights, noisy, max_iter=30)

            # Measure how well we recovered the original
            overlap_before = similarity(noisy, original)
            overlap_after  = similarity(recalled, original)
            perfect        = np.allclose(recalled, original)

            status = "✓ Perfect" if perfect else (
                     "~ Close  " if overlap_after > 0.8 else
                     "✗ Failed ")

            print(f"  {name:<10} | overlap before: {overlap_before:+.2f} "
                  f"→ after: {overlap_after:+.2f} | {status} "
                  f"({len(history)-1} iters)")

    # ── Show a detailed single recall ─────────────────────────
    print("\n" + "=" * 55)
    print("  DETAILED EXAMPLE: recovering 'Cross' with 30% noise")
    print("=" * 55)

    noisy_cross = add_noise(pattern_cross, noise_level=0.3, seed=7)
    recalled_cross, history, converged = recall(weights, noisy_cross, max_iter=30)

    display_pattern(pattern_cross, W, "Original")
    display_pattern(noisy_cross,   W, "Noisy input (30% flipped bits)")
    display_pattern(recalled_cross, W, "Recalled output")

    print(f"  Converged: {'Yes' if converged else 'No (hit max_iter)'}")
    print(f"  Iterations: {len(history) - 1}")
    print(f"  Energy trace: {[f'{e:.1f}' for _, e in history]}")
    print(f"  Final similarity to original: {similarity(recalled_cross, pattern_cross):+.3f}")

    # ── Capacity experiment ────────────────────────────────────
    print("\n" + "=" * 55)
    print("  CAPACITY EXPERIMENT")
    print("  How many random patterns can we store and recall?")
    print("=" * 55)

    np.random.seed(0)
    results = []
    for num_patterns in range(1, 12):
        patterns = [to_bipolar(np.random.choice([0, 1], N)) for _ in range(num_patterns)]
        W_exp = train(patterns)

        # Test each stored pattern (no noise — pure recall)
        correct = sum(
            np.allclose(recall(W_exp, p, max_iter=20)[0], p)
            for p in patterns
        )
        accuracy = correct / num_patterns
        results.append((num_patterns, accuracy))

        bar = "█" * int(accuracy * 20)
        pad = "·" * (20 - len(bar))
        print(f"  {num_patterns:2d} patterns │ {bar}{pad} {accuracy*100:5.1f}% correct")

    theoretical_limit = int(0.138 * N)
    print(f"\n  Theoretical limit: ~{theoretical_limit} patterns (0.138 × {N} neurons)")

  HOPFIELD NETWORK DEMO

📦 STORING PATTERNS:

  Cross
  ──────────
  ·  ·  █  ·  ·
  ·  ·  █  ·  ·
  █  █  █  █  █
  ·  ·  █  ·  ·
  ·  ·  █  ·  ·


  X shape
  ──────────
  █  ·  ·  ·  █
  ·  █  ·  █  ·
  ·  ·  █  ·  ·
  ·  █  ·  █  ·
  █  ·  ·  ·  █


  Box
  ──────────
  █  █  █  █  █
  █  ·  ·  ·  █
  █  ·  ·  ·  █
  █  ·  ·  ·  █
  █  █  █  █  █

Weight matrix shape: (25, 25)
Network capacity: ~3 patterns for 25 neurons
Patterns stored: 3
Within capacity limit: ✓ Yes

  RECALL TEST

── Noise level: 10% ──────────────────────
  Cross      | overlap before: +0.84 → after: +0.36 | ✗ Failed  (2 iters)
  X shape    | overlap before: +0.84 → after: +0.36 | ✗ Failed  (2 iters)
  Box        | overlap before: +0.84 → after: +1.00 | ✓ Perfect (2 iters)

── Noise level: 20% ──────────────────────
  Cross      | overlap before: +0.44 → after: +0.36 | ✗ Failed  (2 iters)
  X shape    | overlap before: +0.44 → after: +0.36 | ✗ Failed  (3 iters)
  Box        | overlap before: +0.44 → after: +1.0